<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

NETID = 'tgarrio1'
PROJECTS = [
    'movsim_movsim',
    'iml-wg_hepml-livingreview',
    'juliagpu_cuda.jl',
    'soedinglab_hh-suite',
    'jdblischak_workflowr',
    'oscaribv_pyaneti',
    'isl-org_midas',
    'covid-projections_covid-data-model',
    'hive-researchgroup_volumetric-data-interaction',
    'antsx_antspy',
]

# GitHub values collected for Part 1 on 2026-09-24.
GITHUB_INFO = {
    'movsim_movsim': (340, 93, '2026-09-02'),
    'iml-wg_hepml-livingreview': (453, 136, '2026-08-10'),
    'juliagpu_cuda.jl': (1428, 282, '2026-09-24'),
    'soedinglab_hh-suite': (628, 152, '2025-08-12'),
    'jdblischak_workflowr': (907, 107, '2026-07-01'),
    'oscaribv_pyaneti': (53, 16, '2026-06-03'),
    'isl-org_midas': (5418, 726, '2024-08-23'),
    'covid-projections_covid-data-model': (149, 53, '2026-02-15'),
    'hive-researchgroup_volumetric-data-interaction': (3, 1, '2024-10-01'),
    'antsx_antspy': (888, 179, '2026-08-19'),
}

# Activity patterns selected after reviewing the monthly plots.
ACTIVITY_PATTERNS = {
    'movsim_movsim': 'declining',
    'iml-wg_hepml-livingreview': 'irregular',
    'juliagpu_cuda.jl': 'rising',
    'soedinglab_hh-suite': 'declining',
    'jdblischak_workflowr': 'declining',
    'oscaribv_pyaneti': 'declining',
    'isl-org_midas': 'irregular',
    'covid-projections_covid-data-model': 'declining',
    'hive-researchgroup_volumetric-data-interaction': 'irregular',
    'antsx_antspy': 'U-shaped',
}

In [2]:
df = pd.read_csv(f'{NETID}_project_summary.csv', sep=';')
df.columns = ['project', 'commit', 'author', 'time', 'message']
assert set(df['project']) == set(PROJECTS), 'Project list does not match.'
assert not df.duplicated(['project', 'commit']).any()
df['time'] = pd.to_numeric(df['time'], errors='raise').astype('int64')
df['datetime_utc'] = pd.to_datetime(df['time'], unit='s', utc=True)
df['Month'] = df['datetime_utc'].dt.strftime('%Y-%m')
print(f'Loaded {len(df):,} commits across {df["project"].nunique()} projects.')
df.head()

Loaded 40,013 commits across 10 projects.


,project,commit,author,time,message,datetime_utc,Month
0,movsim_movsim,0011be813bdf283cd3f816e3d8af12776a4206f6,rgerm <germ@ralphgerm.de>,1330085492,Added possibility of different traffic composi...,2012-02-24 12:11:32+00:00,2012-02
1,movsim_movsim,00331773b48e4e1d82c03fb907b6b59a923461eb,akegermany <mail@akesting.de>,1424971946,minor code cleaning\n,2015-02-26 17:32:26+00:00,2015-02
2,movsim_movsim,0072d55a83d87c26b6fdc5c287916b1d708dd818,rgerm <germ@ralphgerm.de>,1355560081,Merge branch 'master' of https://github.com/mo...,2012-12-15 08:28:01+00:00,2012-12
3,movsim_movsim,00bc0814f4c1a2e15fc386ab70748c302f4f76ff,Martin Budden <mjbudden@gmail.com>,1304975828,Translated 'troedel' parameter to 'slowdown'.\n,2011-05-09 21:17:08+00:00,2011-05
4,movsim_movsim,00ce8eabd86b058b9d2cda73cb2cd111494ebc96,Ralph Germ <germ@ralphgerm.de>,1535274903,Ignores java flight recorder files.\n,2018-08-26 09:15:03+00:00,2018-08


In [3]:
def build_monthly_series(project_df):
    observed = project_df.groupby('Month').size()
    first_month = pd.Period(observed.index.min(), freq='M')
    last_month = pd.Period(observed.index.max(), freq='M')
    full_months = pd.period_range(first_month, last_month, freq='M').astype(str)
    return (
        observed.reindex(full_months, fill_value=0)
        .rename('#Commits')
        .rename_axis('Month')
        .reset_index()
    )


def find_inactivity_gaps(monthly_counts):
    gaps = []
    gap_start = None
    for index, row in monthly_counts.iterrows():
        if row['#Commits'] == 0 and gap_start is None:
            gap_start = index
        elif row['#Commits'] > 0 and gap_start is not None:
            gap_end = index - 1
            gaps.append({
                'start': monthly_counts.loc[gap_start, 'Month'],
                'end': monthly_counts.loc[gap_end, 'Month'],
                'length': gap_end - gap_start + 1,
            })
            gap_start = None

    if not gaps:
        return {
            'LongestGapStart': 'N/A',
            'LongestGapEnd': 'N/A',
            'LongestGapLength': 0,
            'NumCommitsAfterLastGap': 0,
            'NumberOfGapsInTimeline': 0,
        }

    longest = max(gaps, key=lambda gap: gap['length'])
    longest_end = pd.Period(longest['end'], freq='M')
    periods = pd.PeriodIndex(monthly_counts['Month'], freq='M')
    return {
        'LongestGapStart': longest['start'],
        'LongestGapEnd': longest['end'],
        'LongestGapLength': longest['length'],
        'NumCommitsAfterLastGap': int(
            monthly_counts.loc[periods > longest_end, '#Commits'].sum()
        ),
        'NumberOfGapsInTimeline': sum(gap['length'] >= 3 for gap in gaps),
    }

In [4]:
monthly_data = {}
gap_records = []

for project in PROJECTS:
    project_df = df[df['project'] == project]
    monthly_counts = build_monthly_series(project_df)
    monthly_data[project] = monthly_counts
    monthly_counts.to_csv(
        f'{NETID}_commits_timeseries_{project}.csv', sep=';', index=False
    )

    plot_dates = pd.to_datetime(monthly_counts['Month'] + '-01')
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(plot_dates, monthly_counts['#Commits'], marker='o', markersize=2)
    ax.set_xlabel('Time (YYYY-MM)')
    ax.set_ylabel('Number of Commits')
    ax.set_title(project)
    ax.grid(True, alpha=0.35)
    locator = mdates.AutoDateLocator(minticks=6, maxticks=12)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    fig.tight_layout()
    fig.savefig(f'{NETID}_timeseries_{project}.png', dpi=300)
    plt.close(fig)

    gap_record = {'Project': project}
    gap_record.update(find_inactivity_gaps(monthly_counts))
    gap_records.append(gap_record)

gap_stats = pd.DataFrame(gap_records)
print(f'Created {len(PROJECTS)} monthly CSV files and plots.')
gap_stats

Created 10 monthly CSV files and plots.


,Project,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline
0,movsim_movsim,2019-02,2020-06,17,42,10
1,iml-wg_hepml-livingreview,2022-02,2022-02,1,544,0
2,juliagpu_cuda.jl,2013-10,2014-01,4,14305,3
3,soedinglab_hh-suite,2023-09,2024-09,13,14,6
4,jdblischak_workflowr,2024-03,2024-10,8,23,6
5,oscaribv_pyaneti,2019-03,2019-10,8,56,9
6,isl-org_midas,2024-10,2025-06,9,10,5
7,covid-projections_covid-data-model,N/A,N/A,0,0,0
8,hive-researchgroup_volumetric-data-interaction,2023-01,2023-02,2,1284,0
9,antsx_antspy,2017-12,2018-04,5,1966,1


# Interpretation

The classifications below describe the overall trajectories in the monthly plots. Gap counts include only continuous inactive periods lasting at least three months.

- **movsim_movsim — declining:** Activity was highest in 2011–2013, including monthly peaks above 200 commits, and then fell to mostly single-digit activity. It has 10 gaps of at least three months; the longest lasted 17 months from 2019-02 through 2020-06, followed by 42 commits.
- **iml-wg_hepml-livingreview — irregular:** Activity repeatedly jumps between low-volume months and isolated peaks without a sustained direction or fixed cycle. It has no gaps of at least three months; its longest inactive period is only one month, 2022-02.
- **juliagpu_cuda.jl — rising:** Monthly activity grew from near zero in 2013–2015 to sustained high levels, often above 100 commits, from roughly 2019 onward. It has 3 gaps of at least three months; the longest lasted four months from 2013-10 through 2014-01, after which 14,305 commits were made.
- **soedinglab_hh-suite — declining:** Earlier years contain frequent activity and several large bursts, whereas recent years are dominated by very low or zero-commit months. It has 6 gaps of at least three months; the longest lasted 13 months from 2023-09 through 2024-09, followed by 14 commits.
- **jdblischak_workflowr — declining:** The project regularly produced 20–80 commits per month through 2019, but activity dropped sharply after 2020 and has remained sparse. It has 6 gaps of at least three months; the longest lasted eight months from 2024-03 through 2024-10, followed by 23 commits.
- **oscaribv_pyaneti — declining:** High and volatile activity in 2016–2018 gave way to mostly zero or low-single-digit months after 2019. It has 9 gaps of at least three months; the longest lasted eight months from 2019-03 through 2019-10, followed by 56 commits.
- **isl-org_midas — irregular:** The timeline alternates between many zero-commit months and isolated bursts, with no consistent long-term movement or repeating interval. It has 5 gaps of at least three months; the longest lasted nine months from 2024-10 through 2025-06, followed by 10 commits.
- **covid-projections_covid-data-model — declining:** Activity peaked near 1,700 commits per month early in 2020 and then generally fell toward zero by 2024. It has no inactivity gaps because every month between its first and last commit contains activity.
- **hive-researchgroup_volumetric-data-interaction — irregular:** Activity consists of uneven bursts, including large peaks in 2022, 2023, and 2024, separated by low-volume months without a fixed cycle. It has no gaps of at least three months; its longest inactive period lasted two months from 2023-01 through 2023-02.
- **antsx_antspy — U-shaped:** Activity was very high at the beginning, remained comparatively low through the middle years, and rose again with substantial bursts from 2023 onward. It has 1 gap of at least three months; that longest gap lasted five months from 2017-12 through 2018-04, after which 1,966 commits were made.

In [5]:
part1_stats = (
    df.groupby('project')
    .agg(
        ncommits=('commit', 'nunique'),
        nauthors=('author', 'nunique'),
        **{'from': ('time', 'min'), 'to': ('time', 'max')},
    )
    .reset_index()
    .rename(columns={'project': 'Project'})
)
github_stats = pd.DataFrame([
    {
        'Project': project,
        'nstars': GITHUB_INFO[project][0],
        'nforks': GITHUB_INFO[project][1],
        'lastGHCommitDate': GITHUB_INFO[project][2],
    }
    for project in PROJECTS
])
project_stats = (
    part1_stats
    .merge(github_stats, on='Project', validate='one_to_one')
    .merge(gap_stats, on='Project', validate='one_to_one')
)
project_stats['ActivityPattern'] = project_stats['Project'].map(ACTIVITY_PATTERNS)
project_stats['_order'] = project_stats['Project'].map(
    {project: index for index, project in enumerate(PROJECTS)}
)
project_stats = project_stats.sort_values('_order').drop(columns='_order')
project_stats = project_stats[[
    'Project', 'ncommits', 'nauthors', 'from', 'to',
    'nstars', 'nforks', 'lastGHCommitDate',
    'LongestGapStart', 'LongestGapEnd', 'LongestGapLength',
    'NumCommitsAfterLastGap', 'ActivityPattern',
    'NumberOfGapsInTimeline',
]]
project_stats.to_csv(f'{NETID}_project_stats.csv', sep=';', index=False)
project_stats

,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline
7,movsim_movsim,2534,23,1302020322,1762103089,340,93,2026-09-02,2019-02,2020-06,17,42,declining,10
3,iml-wg_hepml-livingreview,877,96,1587584663,1762203246,453,136,2026-08-10,2022-02,2022-02,1,544,irregular,0
6,juliagpu_cuda.jl,14323,381,1378553736,1762624702,1428,282,2026-09-24,2013-10,2014-01,4,14305,rising,3
9,soedinglab_hh-suite,1485,98,1226577471,1755607024,628,152,2025-08-12,2023-09,2024-09,13,14,declining,6
5,jdblischak_workflowr,1399,31,1481147440,1754099317,907,107,2026-07-01,2024-03,2024-10,8,23,declining,6
8,oscaribv_pyaneti,662,5,1453285532,1762449400,53,16,2026-06-03,2019-03,2019-10,8,56,declining,9
4,isl-org_midas,200,41,1561385316,1755784085,5418,726,2024-08-23,2024-10,2025-06,9,10,irregular,5
1,covid-projections_covid-data-model,14427,69,1584466828,1712160037,149,53,2026-02-15,N/A,N/A,0,0,declining,0
2,hive-researchgroup_volumetric-data-interaction,1589,10,1630482898,1727767776,3,1,2024-10-01,2023-01,2023-02,2,1284,irregular,0
0,antsx_antspy,2517,65,1503943258,1773735351,888,179,2026-08-19,2017-12,2018-04,5,1966,U-shaped,1
